# 03 - Feature Engineering

## Objetivo
Crear features derivadas que ayuden a los modelos a detectar patrones pre-falla:
1. **Temporal**: Hora, dia de la semana (codificacion ciclica)
2. **Rolling stats**: Media y std moviles en ventanas de 1h, 3h, 12h
3. **Lags**: Valores pasados (15min, 1h, 3h, 12h atras)
4. **Rate of change**: Velocidad de cambio de metricas
5. **Domain features**: Health score, ratios de fibra optica

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.preprocessor import FiberDataPreprocessor
from src.data.feature_engineer import FiberFeatureEngineer
from src.utils.visualization import apply_style

apply_style()

In [ ]:
# Paso 1: Preprocesar datos
preprocessor = FiberDataPreprocessor()
train_df, val_df, test_df = preprocessor.process_pipeline()

print(f"Train: {train_df.shape}")
print(f"Val: {val_df.shape}")
print(f"Test: {test_df.shape}")

In [ ]:
# Paso 2: Feature Engineering
fe = FiberFeatureEngineer()

train_feat = fe.create_all_features(train_df)
val_feat = fe.create_all_features(val_df)
test_feat = fe.create_all_features(test_df)

feature_names = fe.get_feature_names(train_feat)
print(f"\nTotal features: {len(feature_names)}")
print(f"\nPrimeras 20 features:")
for i, name in enumerate(feature_names[:20]):
    print(f"  {i+1}. {name}")

In [ ]:
# Importancia de features para detectar fallas
# Usamos correlacion con el target como indicador rapido
correlations = train_feat[feature_names + ['fault_in_24h']].corr()['fault_in_24h'].drop('fault_in_24h')
top_features = correlations.abs().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in correlations[top_features.index]]
top_features.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('|Correlacion| con fault_in_24h')
ax.set_title('Top 20 Features por Correlacion con Target')
plt.tight_layout()
plt.show()

print("\nTop 10 features mas correlacionadas:")
for feat, corr_val in correlations[top_features.index].head(10).items():
    print(f"  {feat}: {corr_val:+.4f}")

In [ ]:
# Visualizar health_score por tipo de falla
if 'health_score' in train_feat.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    for fault_type in ['normal', 'physical_cut', 'degradation', 'bad_splice']:
        data = train_feat[train_feat['fault_type'] == fault_type]['health_score']
        if len(data) > 0:
            ax.hist(data, bins=50, alpha=0.5, label=fault_type, density=True)
    ax.set_xlabel('Health Score')
    ax.set_ylabel('Densidad')
    ax.set_title('Distribucion del Health Score por Tipo de Falla')
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Guardar features procesadas
from src.utils.config_loader import get_path

processed_path = get_path('processed_data')
train_feat.to_parquet(processed_path / 'train_features.parquet', index=False)
val_feat.to_parquet(processed_path / 'val_features.parquet', index=False)
test_feat.to_parquet(processed_path / 'test_features.parquet', index=False)

print(f"Features guardadas en {processed_path}")
print(f"  Train: {train_feat.shape}")
print(f"  Val: {val_feat.shape}")
print(f"  Test: {test_feat.shape}")